# Stage 6 — Visualisation

Generate all 7 chart exports to `outputs/figures/`.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import ast
import matplotlib
matplotlib.use("Agg")
sys.path.insert(0, str(Path(".").resolve()))

from src.utils import (
    get_skill_frequency, get_top_skills, compute_placement_lift,
    correlation_matrix, skill_trend_over_time, select_temporal_granularity,
    plot_top_skills, plot_skill_trend, plot_placement_lift,
    plot_salary_distribution, plot_salary_vs_experience,
    plot_skill_boxplot, plot_correlation_heatmap,
)

In [ ]:
df_jobs = pd.read_csv("data/interim/jobs_canonicalized.csv", parse_dates=["date_posted"])
df_placements = pd.read_csv("data/interim/placements_canonicalized.csv")
df_jobs["canonical_skills"] = df_jobs["canonical_skills"].apply(ast.literal_eval)
skill_ds = pd.read_csv("data/processed/skill_demand_supply.csv")
lift_df = pd.read_csv("data/output/placement_lift.csv")
from pathlib import Path
Path("outputs/figures").mkdir(parents=True, exist_ok=True)

## Chart 1 — Top-20 skills

In [ ]:
freq = get_skill_frequency(df_jobs, "canonical_skills")
top20 = get_top_skills(freq, n=20)
plot_top_skills(top20, "outputs/figures/top_skills.png")
print("Saved top_skills.png")

## Chart 2 — Skill trend over time

In [ ]:
granularity = select_temporal_granularity(df_jobs["date_posted"])
trend = skill_trend_over_time(df_jobs, "date_posted", resample_rule=granularity)
plot_skill_trend(trend, "outputs/figures/skill_trend.png")
print("Saved skill_trend.png")

## Chart 3 — Placement lift

In [ ]:
plot_placement_lift(lift_df, "outputs/figures/placement_lift.png")
print("Saved placement_lift.png")

## Chart 4 — Salary distribution

In [ ]:
plot_salary_distribution(df_jobs, "salary_lpa", "outputs/figures/salary_dist.png")
print("Saved salary_dist.png")

## Chart 5 — Salary vs Experience

In [ ]:
plot_salary_vs_experience(df_jobs, "salary_lpa", "experience_min_yrs", "outputs/figures/salary_exp_scatter.png")
print("Saved salary_exp_scatter.png")

## Chart 6 — Skill boxplot by sector

In [ ]:
df_jobs["skill_count"] = df_jobs["canonical_skills"].apply(len)
plot_skill_boxplot(df_jobs, "sector", "skill_count", "outputs/figures/skill_boxplot.png")
print("Saved skill_boxplot.png")

## Chart 7 — Correlation heatmap

In [ ]:
numeric_cols = [c for c in ["years_of_experience","offered_salary_lpa","placed"] if c in df_placements.columns]
corr = correlation_matrix(df_placements, numeric_cols)
plot_correlation_heatmap(corr, "outputs/figures/corr_heatmap.png")
print("Saved corr_heatmap.png")

## Summary

In [ ]:
import os
figs = [f for f in os.listdir("outputs/figures") if f.endswith(".png")]
print(f"{len(figs)} charts in outputs/figures/:", sorted(figs))